# FAO Unified Pipeline

Single entry point for the crop-mapping pipeline, with independently switchable NDVI and
static-image sources. Replaces notebooks 1-3; all logic lives in importable modules in
this folder:

| Module | Role |
|---|---|
| `config.py` | `PipelineConfig`, crop presets, `ModelSource` |
| `model_registry.py` | model resolution + permanent download cache |
| `gee_client.py` | GEE acquisition (grid split, ingest, exports) |
| `gcs_io.py` / `raster_io.py` | GCS downloads (shard-safe), streaming mosaics |
| `inference_workers.py` | Whittaker smoothing + RF inference |
| `static_classify.py` | XGBoost static classification + crop masking |
| `postprocess.py` | sieve + vectorise/export |
| `ndvi_pipeline.py` / `static_pipeline.py` | per-stage source dispatch |
| `pipeline.py` | `run_pipeline(cfg)` -- the whole thing |
| `batch.py` | many districts/crops/years in one run |

`farmdar.sentinel` is **never edited** -- it is synced read-only from the farmdar repo.
Every fix (including the static-date-selection bug) is achieved through how it is called.


In [ ]:
# Pick up edits to the pipeline modules without restarting the kernel.
# Without this, Python caches modules in sys.modules and a re-run of this cell
# keeps executing the code as it was when first imported.
%load_ext autoreload
%autoreload 2


## 1. Config

The only cell you normally edit. The two switches plus their sub-modes cover every
combination:

| `ndvi_source` | `static_source` | sub-mode | Use case |
|---|---|---|---|
| `stac` | `stac` | `stac_static_mode='auto'` | Fully open-source, no GEE (default) |
| `stac` | `stac` | `stac_static_mode='manual'` | Open-source, but you supply the static date(s) |
| `stac` | `gee` | `gee_static_mode='manual_gcs_link'` | Static image you exported by hand from the GEE Code Editor (notebook 3's flow) |
| `stac` | `gee` | `gee_static_mode='api_auto'` | GEE picks + exports the composite by code |
| `stac` | `gee` | `gee_static_mode='api_manual'` | You name the date, GEE exports it by code |
| `gee` | `stac` / `gee` | any | GEE NDVI (notebook 1's flow); pre-2018 years use Landsat 8 |

**Models** (`ndvi_model`, `static_model`) accept three forms:

```python
ndvi_model = "/home/jovyan/FAO/cane/model_files/best_rf_classifier.joblib"   # local path, used as-is
ndvi_model = "gs://bucket/models/best_rf_classifier.joblib"                  # downloaded once, then cached
ndvi_model = {"gcs_uri": "gs://other-bucket/model.joblib",                   # its own credentials
              "gcs_key_path": "/path/to/other_sa_key.json"}
```

A `gs://` model is downloaded once into `model_cache_dir` (default
`~/.cache/fao_pipeline/models`) and read straight from there on every later run.


In [1]:
import logging
from config import build_pipeline_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s", force=True)

cfg = build_pipeline_config(
    crop="cane",                     # 'cane' | 'wheat' | 'spr_maize' | 'rice'  (see config.get_crop_config)
    year="2025",
    district_name="fao_cane_validation_aoi_11",
    aoi_shapefile=r"C:\Work_Work_Work\Python\Scripts\FAO\cane\validation_data\split_aoi_folders\fao_cane_validation_aoi_11\fao_cane_validation_aoi_11.shp",            # default: {base_dir}/{crop}/all_districts_{crop}/{district}/{district}.shp

    # ---- the two independent source switches ----
    ndvi_source="stac",              # 'gee' | 'stac'
    static_source="gee",            # 'stac' | 'gee'
    run_static_model=True,

    # ---- STAC static sub-mode ----
    stac_static_mode="auto",         # 'auto' = cloud-aware date selection | 'manual' = your dates
    # stac_static_dates=["2025-11-02", "2025-11-03"],

    # ---- GEE static sub-mode (only when static_source='gee') ----
    gee_static_mode="manual_gcs_link",      # 'api_auto' | 'api_manual' | 'manual_gcs_link'
    # gee_static_single_date="2025-10-16",
    gee_static_gcs_uri="gs://farmdar_data_catalog/fao_cane_2025/fao_cane_validation_aoi_11/static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16.tif",

    # ---- models: local path, gs:// URI, or {"gcs_uri": ..., "gcs_key_path": ...} ----
    # ndvi_model="gs://farmdar_data_catalog/models/cane/best_rf_classifier.joblib",
    # static_model={"gcs_uri": "gs://other-bucket/xgb_cane_model.json",
    #               "gcs_key_path": "/home/jovyan/keys/other_sa.json"},

    gee_landsat_cutover_year=2018,   # year < this -> Landsat 8 only (never Landsat 7)

    # ---- local-disk cleanup ----
    delete_raw_ndvi_tiles=True,      # raw STAC/GEE NDVI tiles, removed after the mosaic is written
    delete_raw_static_tiles=False,    # raw static tiles + VRT + crop mask

    # ---- run control: each stage writes into its own numbered folder ----
    # "resume" continues the latest run (reusing finished work); "new" starts a clean
    # folder so the same AOI/year can be re-run without renaming anything; or give a
    # run id like "2". Per-stage settings override run_mode.
    run_mode="resume",               # 'new' | 'resume' | '<run id>'
    # ndvi_run_mode="resume",        # e.g. keep the expensive NDVI stage...
    # static_run_mode="new",         # ...while forcing a fresh static stage
    # run_tag="cloudfix",            # optional label appended to new folder names

    # ---- compute / memory ----
    # ndvi_worker_count=None,        # default: 75% of cores
    # ndvi_worker_max_tasks=8,       # recycle each worker after N tiles (1 = most memory-safe)
    # static_worker_count=None,      # default: cores - 1
)

cfg.validate()
print(cfg.summary())


2026-09-02 15:29:17,965 - aoi_io - INFO - AOI path resolved for this platform: C:\Work_Work_Work\Python\Scripts\FAO\cane\validation_data\split_aoi_folders\fao_cane_validation_aoi_11\fao_cane_validation_aoi_11.shp -> /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/fao_cane_validation_aoi_11.shp


crop/year/district : cane / 2025 / fao_cane_validation_aoi_11
AOI                : /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/fao_cane_validation_aoi_11.shp  (from C:\Work_Work_Work\Python\Scripts\FAO\cane\validation_data\split_aoi_folders\fao_cane_validation_aoi_11\fao_cane_validation_aoi_11.shp)
NDVI source        : stac
static source      : gee (manual_gcs_link)
NDVI series window : 2024-12-24 -> 2025-11-17
NDVI inference     : 2025-01-01 -> 2025-11-15
static window      : 2025-10-15 -> 2025-11-15
NDVI model         : gs://farmdar_data_catalog/fao_cane_model_file/fao_cane_rf_model.joblib
static model       : gs://farmdar_data_catalog/fao_cane_model_file/fao_cane_xgb_model.json
model cache        : /home/da638081/.cache/fao_pipeline/models


## 2. Run

`run_pipeline` does everything: resolves models (downloading and caching if needed),
initialises Earth Engine only when this configuration actually needs it, then runs the
NDVI stage, the static stage, and the vector export.


In [2]:
from pipeline import run_pipeline

result = run_pipeline(cfg)

for key, value in result.items():
    print(f"{key:22}: {value}")


2026-09-02 15:29:25,947 - pipeline - INFO - Pipeline configuration:
crop/year/district : cane / 2025 / fao_cane_validation_aoi_11
AOI                : /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/fao_cane_validation_aoi_11.shp  (from C:\Work_Work_Work\Python\Scripts\FAO\cane\validation_data\split_aoi_folders\fao_cane_validation_aoi_11\fao_cane_validation_aoi_11.shp)
NDVI source        : stac
static source      : gee (manual_gcs_link)
NDVI series window : 2024-12-24 -> 2025-11-17
NDVI inference     : 2025-01-01 -> 2025-11-15
static window      : 2025-10-15 -> 2025-11-15
NDVI model         : gs://farmdar_data_catalog/fao_cane_model_file/fao_cane_rf_model.joblib
static model       : gs://farmdar_data_catalog/fao_cane_model_file/fao_cane_xgb_model.json
model cache        : /home/da638081/.cache/fao_pipeline/models
2026-09-02 15:29:25,949 - model_registry - INFO - ndvi_model: cache hit, using /home/da638081/.cache/fao_pipeline/mo

Loading categorical map for strict sieving: fao_cane_validation_aoi_11_rf_classification_map.tif
Phase 1: base sieve filter (removing blobs < 20 px)...
Phase 2: enforcing strict topological encapsulation...


2026-09-02 15:33:59,404 - pipeline - INFO - NDVI stage finished in 4.6 min -> /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/fao_cane_validation_aoi_11_rf_classification_map_sieved_p20.tif


Writing topologically-enforced output to disk...
SUCCESS: sieved raster saved to: /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/fao_cane_validation_aoi_11_rf_classification_map_sieved_p20.tif


2026-09-02 15:34:01,876 - static_pipeline - INFO - Downloading manually-exported static image: gs://farmdar_data_catalog/fao_cane_2025/fao_cane_validation_aoi_11/static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16.tif
2026-09-02 15:34:05,911 - gcs_io - INFO - Downloading static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16.tif (4.67 MB, 5 threads)
2026-09-02 15:34:10,784 - static_classify - INFO - Building crop mask | crop_classes=(1,) | AOI=/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/fao_cane_validation_aoi_11.shp
Building crop mask: 100%|███████████████████████████████████████| 1/1 [00:00<00:00, 36.93block/s]
2026-09-02 15:34:10,881 - static_classify - INFO - Crop mask coverage: 6,361 px (1.57% of the static image grid)
2026-09-02 15:34:10,900 - static_classify - INFO - Grid parity verified: crop mask is 1:1 with the static image.
2026-09-02 15:34:10,912 - static_classify - INFO - Classifying 1 window(s) ac

Loading categorical map for strict sieving: static_mosaic_static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16_Cls.tif
Phase 1: base sieve filter (removing blobs < 20 px)...
Phase 2: enforcing strict topological encapsulation...
Writing topologically-enforced output to disk...


2026-09-02 15:34:25,772 - pipeline - INFO - Static stage finished in 0.4 min -> /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16/static_mosaic_static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16_Cls_sieved_p20.tif
2026-09-02 15:34:25,805 - rasterio._env - WARNING - CPLE_AppDefined in DeprecationWarning: 'Memory' driver is deprecated since GDAL 3.11. Use 'MEM' onwards. Further messages of this type will be suppressed.


SUCCESS: sieved raster saved to: /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16/static_mosaic_static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16_Cls_sieved_p20.tif
1. Vectorising target classes [1]...
  -> Extracted 401 raw polygon features.
  -> Pre-filter dropped 383 sub-threshold polygons before clipping.
2. Loading boundary for clipping...
3. Clip to boundary, explode to singlepart...
4. Computing area (EPSG:32642) and filtering (>= 0.5 acres)...
   Retained 18 features, 15.51 acres, label 1
5. Exporting...


2026-09-02 15:34:28,919 - pyogrio._io - INFO - Created 18 records
2026-09-02 15:34:29,022 - pyogrio._io - INFO - Created 18 records
2026-09-02 15:34:29,157 - pipeline - INFO - Pipeline finished in 5.1 min -> /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16/final_output/fao_cane_validation_aoi_11_cane_2025.gpkg


   -> Saved GPKG: /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16/final_output/fao_cane_validation_aoi_11_cane_2025.gpkg
   -> Saved ZIP (Shapefile): /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/static_cleanup_fao_cane_validation_aoi_11_manual_2025-10-16/final_output/fao_cane_validation_aoi_11_cane_2025.zip
crop                  : cane
year                  : 2025
district              : fao_cane_validation_aoi_11
ndvi_source           : stac
static_source         : gee
output_dir            : /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025
sieved_ndvi_raster    : /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/fao_cane_validation_aoi_11_rf_classificati

## 3. Batch: many districts / crops / years

Jobs run one after another (each already uses every core), Earth Engine is initialised
once for the whole batch, and a failing job is recorded and skipped instead of killing
the run. `run_batch` returns one row per job and can write a summary CSV.


In [4]:
from batch import build_jobs, run_batch

jobs = build_jobs(
    crop="cane",
    year="2025",
    districts=["fao_cane_validation_aoi_11", "fao_cane_validation_aoi_0", "fao_cane_validation_aoi_1"],
    ndvi_source="stac",
    static_source="stac",
    stac_static_mode="auto",
    aoi_path=r"C:\Work_Work_Work\Python\Scripts\FAO\cane\validation_data\split_aoi_folders\{district_name}\{district_name}.shp",
)

# Groups with different settings just concatenate:
# jobs += build_jobs(crop="wheat", year="2021", districts=["Attock"],
#                    static_source="gee", gee_static_mode="manual_gcs_link",
#                    gee_static_gcs_uri="gs://farmdar_data_catalog/.../Attock_2021-Jan-25.tif")

results = run_batch(jobs, continue_on_error=True, results_csv="batch_results.csv")

import pandas as pd
pd.DataFrame(results)


2026-09-02 13:59:58,917 - batch - INFO - Starting batch of 3 job(s).
2026-09-02 13:59:58,917 - batch - INFO - [1/3] cane 2025 fao_cane_validation_aoi_11: starting
2026-09-02 13:59:58,923 - aoi_io - INFO - AOI path resolved for this platform: C:\Work_Work_Work\Python\Scripts\FAO\cane\validation_data\split_aoi_folders\fao_cane_validation_aoi_11\fao_cane_validation_aoi_11.shp -> /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/fao_cane_validation_aoi_11.shp
2026-09-02 13:59:58,981 - pipeline - INFO - Pipeline configuration:
crop/year/district : cane / 2025 / fao_cane_validation_aoi_11
AOI                : /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/fao_cane_validation_aoi_11.shp  (from C:\Work_Work_Work\Python\Scripts\FAO\cane\validation_data\split_aoi_folders\fao_cane_validation_aoi_11\fao_cane_validation_aoi_11.shp)
NDVI source        : stac
static source      : stac 

Loading categorical map for strict sieving: fao_cane_validation_aoi_11_rf_classification_map.tif
Phase 1: base sieve filter (removing blobs < 20 px)...
Phase 2: enforcing strict topological encapsulation...
Writing topologically-enforced output to disk...


2026-09-02 14:04:01,128 - pipeline - INFO - NDVI stage finished in 4.0 min -> /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/fao_cane_validation_aoi_11_rf_classification_map_sieved_p20.tif


SUCCESS: sieved raster saved to: /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/fao_cane_validation_aoi_11_rf_classification_map_sieved_p20.tif


2026-09-02 14:04:03,083 - farmdar.sentinel - INFO - static: 15 scenes on 14 dates in [2025-10-15, 2025-11-15)
2026-09-02 14:04:09,394 - farmdar.sentinel - INFO - static: dates=['2025-10-16'] (anchor=2025-10-16) -> 100.00% of AOI usable
2026-09-02 14:04:09,397 - farmdar.sentinel - INFO - AOI -> 1 tiles @ 10m (tile=0.10deg, bands=['blue', 'green', 'red', 'rededge1', 'nir', 'ndvi'], dates=['2025-10-16'], 8 workers)
2026-09-02 14:04:19,471 - farmdar.sentinel - INFO - [1/1] tile 0001: written (8.7s, 100.0% filled)
2026-09-02 14:04:19,586 - farmdar.sentinel - INFO - DONE 1 tiles in 0.2 min. dates=['2025-10-16'] out_dir=/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/static_staging vrt=/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_11/cane_2025/static_staging/static.vrt
2026-09-02 14:04:19,587 - static_pipeline - INFO - STAC static dates (auto mode): ['2025-10-16']
202

...

2026-09-02 14:15:35,003 - pipeline - INFO - Static stage finished in 0.7 min -> /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_1/cane_2025/16_Oct_2025/static_mosaic_16_Oct_2025_Cls_sieved_p20.tif


Writing topologically-enforced output to disk...
SUCCESS: sieved raster saved to: /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_1/cane_2025/16_Oct_2025/static_mosaic_16_Oct_2025_Cls_sieved_p20.tif
1. Vectorising target classes [1]...
  -> Extracted 553 raw polygon features.
  -> Pre-filter dropped 285 sub-threshold polygons before clipping.
2. Loading boundary for clipping...
3. Clip to boundary, explode to singlepart...
4. Computing area (EPSG:32642) and filtering (>= 0.5 acres)...
   Retained 268 features, 493.71 acres, label 1
5. Exporting...


2026-09-02 14:15:35,524 - pyogrio._io - INFO - Created 268 records
2026-09-02 14:15:35,623 - pyogrio._io - INFO - Created 268 records
2026-09-02 14:15:35,754 - pipeline - INFO - Pipeline finished in 5.3 min -> /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_1/cane_2025/16_Oct_2025/final_output/fao_cane_validation_aoi_1_cane_2025.gpkg
2026-09-02 14:15:35,755 - batch - INFO - [3/3] cane 2025 fao_cane_validation_aoi_1: done in 5.3 min
2026-09-02 14:15:35,756 - batch - INFO - Batch finished: 3 succeeded, 0 failed.
2026-09-02 14:15:35,783 - batch - INFO - Batch summary written to batch_results.csv


   -> Saved GPKG: /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_1/cane_2025/16_Oct_2025/final_output/fao_cane_validation_aoi_1_cane_2025.gpkg
   -> Saved ZIP (Shapefile): /mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/validation_data/split_aoi_folders/fao_cane_validation_aoi_1/cane_2025/16_Oct_2025/final_output/fao_cane_validation_aoi_1_cane_2025.zip


,status,error,crop,year,district,ndvi_source,static_source,output_dir,sieved_ndvi_raster,sieved_static_raster,vector_output,ndvi_minutes,static_minutes,total_minutes
0,ok,None,cane,2025,fao_cane_validation_aoi_11,stac,stac,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,4.0,0.6,4.6
1,ok,None,cane,2025,fao_cane_validation_aoi_0,stac,stac,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,5.1,0.6,5.7
2,ok,None,cane,2025,fao_cane_validation_aoi_1,stac,stac,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,/mnt/c/Work_Work_Work/Python/Scripts/FAO/cane/...,4.6,0.7,5.3


Or from a shell, without opening the notebook:

```bash
python batch.py --jobs jobs.json --results batch_results.csv
```


## 4. Running the stages separately (optional)

`run_pipeline` is the normal path. Call the stages directly only when you want to re-run
one of them -- for example, keep an existing NDVI result and retry just the static model
with a different date.


In [ ]:
# from pathlib import Path

# import model_registry
# from ndvi_pipeline import run_ndvi_pipeline
# from static_pipeline import run_static_pipeline
# from pipeline import default_output_dir

# out_dir = default_output_dir(cfg)
# models = model_registry.resolve_pipeline_models(cfg)

# sieved_ndvi_path = run_ndvi_pipeline(cfg, out_dir, ndvi_model_path=str(models["ndvi_model"]))
# print(sieved_ndvi_path)

# # sieved_static_path = run_static_pipeline(
# #     cfg, out_dir, sieved_ndvi_path, static_model_path=str(models["static_model"]),
# # )
